# Lab 6 — Web Search Agent (Brave Search partner extension)

Adds a **Research Agent** that searches the live web via the [Brave Search API](https://brave.com/search/api/). Web search is a tool wired through **L3 Orchestration** (an AgentCore Gateway MCP Lambda target) that strengthens **L1 Data & Knowledge** with live external retrieval. Same pattern as the Order/Refund agents: **Lambda tool → Gateway target → specialist A2A agent**, so it inherits L4 masking and L5 tracing.

> **Prerequisite:** run the base workshop's **L2** and **L3 `1_setup_resources`** notebooks first (this lab reads the model id, Gateway, Cognito, and Registry from SSM). Region: **us-west-2**.

> **AWS-native alternative:** AgentCore also offers a first-party managed **Web Search Tool** (a built-in Gateway connector, `connectorId: "web-search"`, backed by an Amazon-operated index, `us-east-1` only today). This lab uses **Brave** to demonstrate the *partner-extension pattern* and to provide web search in-region (us-west-2). Both occupy the same Gateway MCP slot; see the module README for the full native-vs-partner contrast.


In [ ]:
%pip install --quiet boto3==1.43.0 strands-agents==1.43.0 strands-agents-tools==0.2.0 bedrock-agentcore==1.14.0 bedrock-agentcore-starter-toolkit==0.3.6 requests

## 1. Configuration & prerequisite check

Reads base-workshop resource IDs from SSM and fails clearly if they are missing.

In [ ]:
import json, time, os, io, zipfile, uuid, shutil
import boto3

REGION = os.environ.get("AWS_DEFAULT_REGION") or boto3.session.Session().region_name or "us-west-2"
SSM_PREFIX = "/anycompany/agentcore"
NOTEBOOK_DIR = os.getcwd()
# Locate the Brave module 'code' dir relative to this notebook. Works in the
# repo layout (aws-brave/workshop/l3-orchestration -> ../../code) and in the
# flattened participant layout (l3-orchestration -> ../code).
_CAND = [os.path.abspath(os.path.join(NOTEBOOK_DIR, p)) for p in ("../code", "../../code")]
CODE_DIR = next((c for c in _CAND if os.path.isdir(os.path.join(c, "web_search_lambda"))), _CAND[0])

ssm = boto3.client("ssm", region_name=REGION)
iam = boto3.client("iam")
sts = boto3.client("sts", region_name=REGION)
lambda_client = boto3.client("lambda", region_name=REGION)
ecr = boto3.client("ecr", region_name=REGION)
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
ACCOUNT_ID = sts.get_caller_identity()["Account"]


def get_ssm(name, default=None):
    try:
        return ssm.get_parameter(Name=name, WithDecryption=True)["Parameter"]["Value"]
    except ssm.exceptions.ParameterNotFound:
        return default


# --- Prerequisite check: inform (don't crash) if earlier labs have not run ---
_required = {
    "model_id": "L2  \u2192  l2-inference/1_pluggable_inference_layer.ipynb",
    "gateway_id": "L3  \u2192  l3-orchestration/1_setup_resources.ipynb",
    "gateway_url": "L3  \u2192  l3-orchestration/1_setup_resources.ipynb",
    "gateway_role_arn": "L3  \u2192  l3-orchestration/1_setup_resources.ipynb",
    "registry_id": "L3  \u2192  l3-orchestration/1_setup_resources.ipynb",
    "cognito_client_id": "L3  \u2192  l3-orchestration/1_setup_resources.ipynb",
}
_vals = {k: get_ssm(f"{SSM_PREFIX}/{k}") for k in _required}
_missing = {k: _required[k] for k, v in _vals.items() if v is None}
if _missing:
    print("=" * 72)
    print("  Prerequisites not met - please run these notebook(s) first, in order:")
    print()
    for nb in sorted(set(_missing.values())):
        print("     -", nb)
    print()
    print("  (missing config: " + ", ".join(sorted(_missing)) + ")")
    print()
    print("  Then re-run this cell. Nothing was created.")
    print("=" * 72)
    raise SystemExit("Prerequisites not met - run the notebook(s) listed above, then re-run this cell.")

GATEWAY_ID = _vals["gateway_id"]
GATEWAY_URL = _vals["gateway_url"]
GATEWAY_ROLE_ARN = _vals["gateway_role_arn"]
REGISTRY_ID = _vals["registry_id"]
MODEL_ID = _vals["model_id"]
print(f"Account {ACCOUNT_ID} | Region {REGION}")
print(f"Gateway {GATEWAY_ID} | Registry {REGISTRY_ID}")
print("Prerequisites OK.")


## 2. Enter your Brave API key

Paste your Brave Search API key over the placeholder, then run the cell. It is stored in **Secrets Manager** (`brave/search-api-key`); the Lambda reads it at runtime.

> ⚠️ Do not save/commit this notebook with your real key pasted in.

In [ ]:
SECRET_NAME = "brave/search-api-key"

# 👉 Paste your Brave Search API key between the quotes below, then run this cell.
BRAVE_API_KEY = ""

if not BRAVE_API_KEY.strip():
    raise SystemExit("No key entered - paste your Brave API key into the BRAVE_API_KEY line above, then re-run this cell.")
if not (BRAVE_API_KEY.startswith("BSA") and len(BRAVE_API_KEY) > 25):
    print("Note: Brave keys normally start with 'BSA' and are ~31 chars - double-check you pasted the full key.")

sm = boto3.client("secretsmanager", region_name=REGION)
_secret = json.dumps({"api_key": BRAVE_API_KEY})
try:
    sm.create_secret(Name=SECRET_NAME, SecretString=_secret)
    print(f"Created secret {SECRET_NAME}")
except sm.exceptions.ResourceExistsException:
    sm.put_secret_value(SecretId=SECRET_NAME, SecretString=_secret)
    print(f"Updated existing secret {SECRET_NAME}")
print(f"Stored a {len(BRAVE_API_KEY)}-char key.")
del BRAVE_API_KEY, _secret


## 3. Create the Lambda execution role (least privilege)

Allows only `GetSecretValue` on the Brave secret, plus basic Lambda logging.

In [ ]:
LAMBDA_ROLE_NAME = "AnyCompanyBraveWebSearchLambdaRole"
trust = json.dumps({"Version": "2012-10-17", "Statement": [
    {"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]})
try:
    LAMBDA_ROLE_ARN = iam.create_role(RoleName=LAMBDA_ROLE_NAME, AssumeRolePolicyDocument=trust,
                                      Description="Exec role for Brave web-search Lambda")["Role"]["Arn"]
    print("Created role:", LAMBDA_ROLE_ARN)
except iam.exceptions.EntityAlreadyExistsException:
    LAMBDA_ROLE_ARN = iam.get_role(RoleName=LAMBDA_ROLE_NAME)["Role"]["Arn"]
    print("Role exists:", LAMBDA_ROLE_ARN)
iam.attach_role_policy(RoleName=LAMBDA_ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
iam.put_role_policy(RoleName=LAMBDA_ROLE_NAME, PolicyName="BraveSecretRead",
    PolicyDocument=json.dumps({"Version": "2012-10-17", "Statement": [
        {"Effect": "Allow", "Action": "secretsmanager:GetSecretValue", "Resource": SECRET_ARN}]}))
print("Waiting for IAM propagation..."); time.sleep(10)

## 4. Package & deploy the `anycompany_brave_web_search` Lambda

Zips the stdlib-only client + handler from `../../code/web_search_lambda/`.

In [ ]:
LAMBDA_NAME = "anycompany_brave_web_search"
SRC = os.path.join(CODE_DIR, "web_search_lambda")
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in ("lambda_function.py", "brave_client.py"):
        z.write(os.path.join(SRC, fn), arcname=fn)
code_bytes = buf.getvalue()
env = {"Variables": {"BRAVE_SECRET_ID": SECRET_NAME}}
try:
    LAMBDA_ARN = lambda_client.create_function(
        FunctionName=LAMBDA_NAME, Runtime="python3.12", Role=LAMBDA_ROLE_ARN,
        Handler="lambda_function.lambda_handler", Code={"ZipFile": code_bytes},
        Timeout=15, Environment=env)["FunctionArn"]
    print("Created function:", LAMBDA_ARN)
except lambda_client.exceptions.ResourceConflictException:
    lambda_client.update_function_code(FunctionName=LAMBDA_NAME, ZipFile=code_bytes)
    lambda_client.get_waiter("function_updated").wait(FunctionName=LAMBDA_NAME)
    lambda_client.update_function_configuration(FunctionName=LAMBDA_NAME, Environment=env, Timeout=15)
    LAMBDA_ARN = lambda_client.get_function_configuration(FunctionName=LAMBDA_NAME)["FunctionArn"]
    print("Updated function:", LAMBDA_ARN)
lambda_client.get_waiter("function_active_v2").wait(FunctionName=LAMBDA_NAME)
print("Lambda active.")

## 5. Register the Gateway target `brave-web-search`

Grants the Gateway role invoke on the Lambda, then registers the MCP target with the tool schema.

In [ ]:
iam.put_role_policy(RoleName=GATEWAY_ROLE_ARN.split("/")[-1], PolicyName="InvokeBraveLambda",
    PolicyDocument=json.dumps({"Version": "2012-10-17", "Statement": [
        {"Effect": "Allow", "Action": "lambda:InvokeFunction", "Resource": LAMBDA_ARN}]}))
time.sleep(8)
with open(os.path.join(CODE_DIR, "research_agent/tool_schemas.json")) as fh:
    TOOL_SCHEMAS = json.load(fh)
TARGET_NAME = "brave-web-search"
try:
    TARGET_ID = agentcore_control.create_gateway_target(
        gatewayIdentifier=GATEWAY_ID, name=TARGET_NAME,
        description="Brave web search (live external retrieval) backed by Lambda",
        targetConfiguration={"mcp": {"lambda": {"lambdaArn": LAMBDA_ARN,
            "toolSchema": {"inlinePayload": TOOL_SCHEMAS}}}},
        credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}])["targetId"]
    print("Created gateway target:", TARGET_ID)
except agentcore_control.exceptions.ConflictException:
    tgts = agentcore_control.list_gateway_targets(gatewayIdentifier=GATEWAY_ID)
    TARGET_ID = next(t["targetId"] for t in tgts.get("items", tgts.get("targets", [])) if t.get("name") == TARGET_NAME)
    print("Target exists:", TARGET_ID)
for _ in range(20):
    if agentcore_control.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)["status"] == "READY":
        break
    time.sleep(5)
print("Gateway target READY -> tool: brave-web-search___web_search")

## 6. Smoke test — raw MCP `tools/call`

Calls the tool through the Gateway (Cognito auth) to confirm live Brave results before involving the agent.

In [ ]:
import urllib.request
cog = boto3.client("cognito-idp", region_name=REGION)
tok = cog.initiate_auth(ClientId=get_ssm(f"{SSM_PREFIX}/cognito_client_id"),
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": "gold_customer", "PASSWORD": get_ssm(f"{SSM_PREFIX}/user_password")}
    )["AuthenticationResult"]["IdToken"]
req = {"jsonrpc": "2.0", "method": "tools/call", "id": str(uuid.uuid4()),
       "params": {"name": "brave-web-search___web_search", "arguments": {"query": "latest AWS news", "count": 3}}}
r = urllib.request.Request(GATEWAY_URL, data=json.dumps(req).encode(),
    headers={"Authorization": f"Bearer {tok}", "Content-Type": "application/json"})
resp = json.loads(urllib.request.urlopen(r, timeout=40).read())
payload = json.loads(resp["result"]["content"][0]["text"])
print("status:", payload["status"])
if payload.get("status") != "success":
    print("message:", payload.get("message"))
for i, it in enumerate(payload.get("results", []), 1):
    print(f"  {i}. {it['title']}  {it['url']}")

## 7. Deploy the Research Agent to AgentCore Runtime

Creates the runtime role and builds/deploys the A2A agent via the starter toolkit (CodeBuild — no local Docker).

In [ ]:
RUNTIME_ROLE_NAME = "AgentCoreResearchAgentA2ARole"
rt_trust = json.dumps({"Version": "2012-10-17", "Statement": [
    {"Effect": "Allow", "Principal": {"Service": "bedrock-agentcore.amazonaws.com"}, "Action": "sts:AssumeRole"}]})
try:
    RUNTIME_ROLE_ARN = iam.create_role(RoleName=RUNTIME_ROLE_NAME, AssumeRolePolicyDocument=rt_trust,
                                       Description="Runtime role for Research A2A Agent")["Role"]["Arn"]
except iam.exceptions.EntityAlreadyExistsException:
    RUNTIME_ROLE_ARN = iam.get_role(RoleName=RUNTIME_ROLE_NAME)["Role"]["Arn"]
iam.put_role_policy(RoleName=RUNTIME_ROLE_NAME, PolicyName="ResearchAgentRuntimeAccess",
    PolicyDocument=json.dumps({"Version": "2012-10-17", "Statement": [
        {"Sid": "Bedrock", "Effect": "Allow", "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"], "Resource": ["arn:aws:bedrock:*::foundation-model/*", "arn:aws:bedrock:*:*:inference-profile/*"]},
        {"Sid": "ECRPublicAuth", "Effect": "Allow", "Action": ["ecr-public:GetAuthorizationToken", "sts:GetServiceBearerToken", "ecr:GetAuthorizationToken"], "Resource": "*"},
        {"Sid": "ECRImage", "Effect": "Allow", "Action": ["ecr:BatchGetImage", "ecr:GetDownloadUrlForLayer", "ecr:BatchCheckLayerAvailability"], "Resource": "arn:aws:ecr:*:*:repository/bedrock-agentcore-*"},
        {"Sid": "XRay", "Effect": "Allow", "Action": ["xray:PutTraceSegments", "xray:PutTelemetryRecords", "xray:GetSamplingRules", "xray:GetSamplingTargets"], "Resource": "*"},
        {"Sid": "Metrics", "Effect": "Allow", "Action": "cloudwatch:PutMetricData", "Resource": "*", "Condition": {"StringEquals": {"cloudwatch:namespace": "bedrock-agentcore"}}},
        {"Sid": "Logs", "Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents", "logs:DescribeLogStreams", "logs:DescribeLogGroups"], "Resource": f"arn:aws:logs:*:{ACCOUNT_ID}:log-group:*"},
        {"Sid": "WorkloadIdentity", "Effect": "Allow", "Action": ["bedrock-agentcore:GetWorkloadAccessToken", "bedrock-agentcore:GetWorkloadAccessTokenForJWT"], "Resource": [f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:workload-identity-directory/default", f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:workload-identity-directory/default/workload-identity/*"]},
        {"Sid": "SSM", "Effect": "Allow", "Action": ["ssm:GetParameter"], "Resource": f"arn:aws:ssm:*:*:parameter{SSM_PREFIX}/*"}]}))
print("Runtime role ready:", RUNTIME_ROLE_ARN); time.sleep(10)

# Pre-create the ECR repo so the first CodeBuild push does not race repo creation
try:
    ecr.create_repository(repositoryName="bedrock-agentcore-research_agent_a2a")
except ecr.exceptions.RepositoryAlreadyExistsException:
    pass

AGENT_DIR = os.path.join(CODE_DIR, "research_agent")
shutil.copy(os.path.join(CODE_DIR, "requirements_research_a2a.txt"),
            os.path.join(AGENT_DIR, "requirements_research_a2a.txt"))
os.chdir(AGENT_DIR)
for f in (".bedrock_agentcore.yaml", "Dockerfile"):
    if os.path.exists(f):
        os.remove(f)
from bedrock_agentcore_starter_toolkit import Runtime
agentcore_rt = Runtime()
agentcore_rt.configure(entrypoint="research_agent_a2a.py", execution_role=RUNTIME_ROLE_ARN,
    auto_create_ecr=True, requirements_file="requirements_research_a2a.txt", region=REGION,
    agent_name="research_agent_a2a", protocol="A2A")
launch_result = agentcore_rt.launch(auto_update_on_conflict=True)
RESEARCH_AGENT_ARN = launch_result.agent_arn
os.chdir(NOTEBOOK_DIR)
print("Launched:", RESEARCH_AGENT_ARN)
for _ in range(90):
    st = agentcore_rt.status().endpoint["status"]
    if st in ("ACTIVE", "READY", "FAILED"):
        break
    time.sleep(10)
print("Deployment status:", st)
ssm.put_parameter(Name=f"{SSM_PREFIX}/research_agent_arn", Value=RESEARCH_AGENT_ARN, Type="String", Overwrite=True)

## 8. Register the Research Agent in the Agent Registry

Fetches the A2A agent card and registers it so the orchestrator can discover it dynamically.

In [ ]:
import requests
from urllib.parse import quote
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

escaped = quote(RESEARCH_AGENT_ARN, safe="")
card_url = f"https://bedrock-agentcore.{REGION}.amazonaws.com/runtimes/{escaped}/invocations/.well-known/agent-card.json"
creds = boto3.Session(region_name=REGION).get_credentials().get_frozen_credentials()
aws_req = AWSRequest(method="GET", url=card_url,
    headers={"X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": str(uuid.uuid4())})
SigV4Auth(creds, "bedrock-agentcore", REGION).add_auth(aws_req)
agent_card = requests.get(card_url, headers=dict(aws_req.headers), timeout=40).json()
print("Agent card:", agent_card.get("name"))

RECORD_NAME = "research_agent_a2a_record"
try:
    reg = agentcore_control.create_registry_record(registryId=REGISTRY_ID, name=RECORD_NAME,
        description="Research Agent — current-info questions via Brave web search.",
        descriptorType="A2A", recordVersion="1.0",
        descriptors={"a2a": {"agentCard": {"inlineContent": json.dumps(agent_card)}}})
    RECORD_ID = reg["recordArn"].rsplit("/", 1)[-1]
    print("Created registry record:", RECORD_ID)
except agentcore_control.exceptions.ConflictException:
    recs = agentcore_control.list_registry_records(registryId=REGISTRY_ID)
    RECORD_ID = next(r["recordId"] for r in recs.get("registryRecords", recs.get("items", [])) if r.get("name") == RECORD_NAME)
    print("Record exists:", RECORD_ID)
for _ in range(20):
    rec = agentcore_control.get_registry_record(registryId=REGISTRY_ID, recordId=RECORD_ID)
    if rec["status"] not in ("CREATING", "UPDATING"):
        break
    time.sleep(2)
agentcore_control.submit_registry_record_for_approval(registryId=REGISTRY_ID, recordId=RECORD_ID)
print("Submitted for approval. Status:", rec["status"])

## 9. Test the Research Agent directly (A2A)

Invokes the deployed agent with a current-info question; it calls Brave via the Gateway and answers with citations.

In [ ]:
USER_PASSWORD = get_ssm(f"{SSM_PREFIX}/user_password")
test_token = boto3.client("cognito-idp", region_name=REGION).initiate_auth(
    ClientId=get_ssm(f"{SSM_PREFIX}/cognito_client_id"), AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": "gold_customer", "PASSWORD": USER_PASSWORD})["AuthenticationResult"]["IdToken"]
invoke_url = f"https://bedrock-agentcore.{REGION}.amazonaws.com/runtimes/{quote(RESEARCH_AGENT_ARN, safe='')}/invocations"
question = "What are the latest AWS news headlines? Please cite your sources."
a2a = json.dumps({"jsonrpc": "2.0", "method": "message/send", "id": str(uuid.uuid4()),
    "params": {"message": {"role": "user", "parts": [{"kind": "text", "text": question}], "messageId": str(uuid.uuid4())}}})
creds = boto3.Session(region_name=REGION).get_credentials().get_frozen_credentials()
headers = {"Content-Type": "application/json",
           "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": str(uuid.uuid4()),
           "Authorization": f"Bearer {test_token}"}
aws_req = AWSRequest(method="POST", url=invoke_url, data=a2a, headers=headers)
SigV4Auth(creds, "bedrock-agentcore", REGION).add_auth(aws_req)
print(f"Q: {question}\n")
resp = requests.post(invoke_url, data=a2a, headers=dict(aws_req.headers), timeout=120)
print("HTTP", resp.status_code)
try:
    rj = resp.json()
    for artifact in rj.get("result", {}).get("artifacts", []):
        for part in artifact.get("parts", []):
            if part.get("kind") == "text":
                print(part["text"][:3000])
except json.JSONDecodeError:
    print(resp.text[:2000])

## 10. Publish resource IDs to SSM

In [ ]:
for k, v in {"brave_web_search_lambda_arn": LAMBDA_ARN,
             "brave_web_search_gateway_target_id": TARGET_ID,
             "research_agent_arn": RESEARCH_AGENT_ARN,
             "research_agent_record_id": RECORD_ID}.items():
    ssm.put_parameter(Name=f"{SSM_PREFIX}/{k}", Value=v, Type="String", Overwrite=True)
print("Published Brave resource IDs to SSM.")

## 11. Cleanup

Deletes everything this lab created. Run when you are done.

In [ ]:
def _safe(desc, fn, *a, **k):
    try:
        fn(*a, **k); print("deleted:", desc)
    except Exception as e:
        print("skip", desc, "-", str(e)[:80])

_safe("registry record", agentcore_control.delete_registry_record, registryId=REGISTRY_ID, recordId=RECORD_ID)
_safe("agent runtime", agentcore_control.delete_agent_runtime, agentRuntimeId=RESEARCH_AGENT_ARN.rsplit("/", 1)[-1])
_safe("gateway target", agentcore_control.delete_gateway_target, gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
_safe("lambda", lambda_client.delete_function, FunctionName=LAMBDA_NAME)
_safe("lambda role policy", iam.delete_role_policy, RoleName=LAMBDA_ROLE_NAME, PolicyName="BraveSecretRead")
_safe("lambda role managed", iam.detach_role_policy, RoleName=LAMBDA_ROLE_NAME, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
_safe("lambda role", iam.delete_role, RoleName=LAMBDA_ROLE_NAME)
_safe("runtime role policy", iam.delete_role_policy, RoleName=RUNTIME_ROLE_NAME, PolicyName="ResearchAgentRuntimeAccess")
_safe("runtime role", iam.delete_role, RoleName=RUNTIME_ROLE_NAME)
_safe("gateway invoke policy", iam.delete_role_policy, RoleName=GATEWAY_ROLE_ARN.split("/")[-1], PolicyName="InvokeBraveLambda")
_safe("secret", sm.delete_secret, SecretId=SECRET_NAME, ForceDeleteWithoutRecovery=True)
for k in ("brave_web_search_lambda_arn", "brave_web_search_gateway_target_id", "research_agent_arn", "research_agent_record_id"):
    _safe(f"ssm {k}", ssm.delete_parameter, Name=f"{SSM_PREFIX}/{k}")
print("Cleanup complete.")